<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/etapa-02-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.md.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 18: Redes Multicamada e Grafos Logísticos de Insumos e Produtos

## 1. Fundamentos Matemáticos: DAGs e Redes Multicamada no SCADA

No ecossistema **SCADA-Core / Visão-AGV**, precisamos mapear não apenas por onde o veículo autônomo anda, mas também a sequência de processos do que ele está transportando. Para evitar falhas críticas de modelagem (como o motor tentar enviar um AGV por dentro de um reator químico ou inverter a ordem de fabricação de um fertilizante), adotamos o conceito de **Redes Multicamada** (*Multilayer Networks*).

Modelaremos a fábrica em dois grafos sobrepostos:
1. **$G_M$ (Grafo de Materiais):** Modela o fluxo de transformação físico-química. É um **Grafo Acíclico Dirigido (DAG)**, pois as etapas de produção são irreversíveis. Utilizaremos o Algoritmo de Kahn (Ordenação Topológica) para auditar a viabilidade do processo.
2. **$G_V$ (Grafo de Veículos):** Modela a malha de circulação física (corredores, docas). Contém ciclos naturais (o AGV pode dar voltas no galpão). Utilizaremos o Algoritmo de Dijkstra para o roteamento do veículo.

Essa separação garante que aplicaremos o algoritmo correto para a camada correspondente da planta industrial.

In [1]:
import heapq
from typing import Dict, List, Tuple

class GrafoLogisticoSCADA:
    """
    Motor SCADA baseado em Redes Multicamada para gestão logística.
    Gerencia simultaneamente o fluxo lógico de materiais (GM) e as rotas físicas do AGV (GV).
    """
    def __init__(self):
        # GM: Grafo de Materiais (DAG - Sem pesos de distância)
        self.gm_adj: Dict[str, List[str]] = {}
        self.gm_in_degree: Dict[str, int] = {}

        # GV: Grafo de Veículos/AGV (Ponderado com distância em metros)
        self.gv_adj: Dict[str, List[Tuple[str, float]]] = {}

    # ==========================================
    # CAMADA DE MATERIAIS (G_M) - FLUXO DO PROCESSO
    # ==========================================
    def adicionar_etapa_processo(self, origem: str, destino: str):
        """Adiciona uma transição irreversível no fluxo de produção de fertilizantes."""
        if origem not in self.gm_adj: self.gm_adj[origem] = []
        if destino not in self.gm_adj: self.gm_adj[destino] = []

        self.gm_adj[origem].append(destino)
        # Atualiza o grau de entrada (pré-requisitos) da etapa destino
        self.gm_in_degree[destino] = self.gm_in_degree.get(destino, 0) + 1
        if origem not in self.gm_in_degree: self.gm_in_degree[origem] = 0

    def auditar_ordem_topologica(self) -> List[str]:
        """
        Algoritmo de Kahn: Valida se o processo de produção é um DAG viável.
        Retorna a sequência lógica de produção ou acusa erro de ciclo.
        """
        fila = [no for no, grau in self.gm_in_degree.items() if grau == 0]
        ordem = []
        in_degree_temp = self.gm_in_degree.copy()

        while fila:
            u = fila.pop(0)
            ordem.append(u)
            for v in self.gm_adj.get(u, []):
                in_degree_temp[v] -= 1
                if in_degree_temp[v] == 0:
                    fila.append(v)

        if len(ordem) != len(self.gm_adj):
            return ["[ERRO DE ENGENHARIA] Ciclo detectado! O fluxo material contém etapas reversas."]
        return ordem

    # ==========================================
    # CAMADA DE VEÍCULOS (G_V) - ROTEAMENTO AGV
    # ==========================================
    def adicionar_rota_agv(self, u: str, v: str, distancia_m: float, bidirecional: bool = True):
        """Mapeia um corredor transitável pela frota de AGVs."""
        if u not in self.gv_adj: self.gv_adj[u] = []
        if v not in self.gv_adj: self.gv_adj[v] = []

        self.gv_adj[u].append((v, distancia_m))
        if bidirecional:
            self.gv_adj[v].append((u, distancia_m))

    def calcular_rota_agv_dijkstra(self, origem: str, destino: str) -> Tuple[List[str], float]:
        """Calcula o caminho físico mais curto evitando obstáculos."""
        distancias = {node: float('inf') for node in self.gv_adj}
        predecessores = {node: None for node in self.gv_adj}
        distancias[origem] = 0.0
        min_heap = [(0.0, origem)]

        while min_heap:
            dist_atual, u = heapq.heappop(min_heap)
            if dist_atual > distancias[u]: continue
            if u == destino: break

            for vizinho, peso in self.gv_adj.get(u, []):
                dist_alt = dist_atual + peso
                if dist_alt < distancias[vizinho]:
                    distancias[vizinho] = dist_alt
                    predecessores[vizinho] = u
                    heapq.heappush(min_heap, (dist_alt, vizinho))

        caminho, passo = [], destino
        while passo:
            caminho.append(passo)
            passo = predecessores[passo]

        return caminho[::-1], distancias[destino]

## 2. Instanciação e Simulação do Modelo Multicamada

Abaixo, populamos e testamos as duas camadas independentes. Primeiro, validamos o sequenciamento de transformação de matéria-prima em produto acabado ($G_M$). Em seguida, mapeamos o pátio logístico para calcular a rota do AGV que transportará os paletes prontos ($G_V$).

In [2]:
scada = GrafoLogisticoSCADA()

# ---------------------------------------------------------
# 1. Configurando GM (Grafo de Materiais - Processos)
# ---------------------------------------------------------
print("=== AUDITORIA DE PROCESSO (CAMADA G_M) ===")
scada.adicionar_etapa_processo("Portaria_Insumos", "Box_Ureia")
scada.adicionar_etapa_processo("Box_Ureia", "Moega_Recepcao")
scada.adicionar_etapa_processo("Moega_Recepcao", "Silos_Dosagem")
scada.adicionar_etapa_processo("Silos_Dosagem", "Granulacao")
scada.adicionar_etapa_processo("Granulacao", "Ensacamento")
scada.adicionar_etapa_processo("Ensacamento", "Estoque_NPK")

ordem_producao = scada.auditar_ordem_topologica()
print(f"Sequência Lógica Validada: \n{' -> '.join(ordem_producao)}\n")

# ---------------------------------------------------------
# 2. Configurando GV (Grafo de Veículos - Planta Física)
# ---------------------------------------------------------
print("=== ROTEAMENTO FÍSICO DO AGV (CAMADA G_V) ===")
scada.adicionar_rota_agv("Portaria", "Galpao_A", 50.0)
scada.adicionar_rota_agv("Galpao_A", "Moega_Fisica", 30.0)
scada.adicionar_rota_agv("Galpao_A", "Corredor_Externo", 45.0)
scada.adicionar_rota_agv("Corredor_Externo", "Docas_Saida", 60.0)
scada.adicionar_rota_agv("Moega_Fisica", "Galpao_B_Ensaque", 40.0)
scada.adicionar_rota_agv("Galpao_B_Ensaque", "Docas_Saida", 20.0)

# Simulando uma ordem de movimentação para o AGV Logístico
origem_agv = "Galpao_A"
destino_agv = "Docas_Saida"

rota_veiculo, dist_total = scada.calcular_rota_agv_dijkstra(origem_agv, destino_agv)

print(f"Ordem de Movimentação: {origem_agv} até {destino_agv}")
print(f"Rota Mínima para o Veículo: {' -> '.join(rota_veiculo)}")
print(f"Distância Total Percorrida: {dist_total:.1f} metros")

=== AUDITORIA DE PROCESSO (CAMADA G_M) ===
Sequência Lógica Validada: 
Portaria_Insumos -> Box_Ureia -> Moega_Recepcao -> Silos_Dosagem -> Granulacao -> Ensacamento -> Estoque_NPK

=== ROTEAMENTO FÍSICO DO AGV (CAMADA G_V) ===
Ordem de Movimentação: Galpao_A até Docas_Saida
Rota Mínima para o Veículo: Galpao_A -> Moega_Fisica -> Galpao_B_Ensaque -> Docas_Saida
Distância Total Percorrida: 90.0 metros
